Number of classes: 28
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.059280 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 76500
[LightGBM] [Info] Number of data points in the train set: 98038, number of used features: 300
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332204
[LightGBM] [Info] Start training from score -3.332205
[LightGBM] [Info] Start train

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:55:07] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Ensemble Validation Accuracy: 0.7853457172342622
Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.57      0.31      0.40        13
           4       0.60      0.70      0.65        46
           5       0.91      0.97      0.94       875
           6       0.91      0.95      0.93       106
           7       0.62      0.38      0.47        21
           8       0.76      0.79      0.77        98
           9       0.00      0.00      0.00         5
          10       0.70      0.85      0.77       208
          11       0.64      0.75      0.69        12
          12       0.47      0.52      0.50        88
          13       0.00      0.00      0.00        12
          14       0.23      0.06      0.09        52
          15       0.80      0.80      0.80         5


/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Weighted Average Loss (Validation): 0.007880784679587374


In [2]:
# ------------------------------------------------------------------
# 0.  If the ensemble is still in memory just save it  -------------
# ------------------------------------------------------------------
import joblib   # part of the scikit‑learn stack
joblib.dump(ensemble, "vote_ensemble.pkl")      # ≈ a few MB on disk
print("Model saved to vote_ensemble.pkl")

# ------------------------------------------------------------------
# 1.  Later / in another notebook: reload the model ----------------
# ------------------------------------------------------------------
import joblib, pandas as pd
ensemble = joblib.load("vote_ensemble.pkl")

# ------------------------------------------------------------------
# 2.  Load the *unseen* test features ------------------------------
# ------------------------------------------------------------------
X_test = pd.read_csv("X_test_1.csv")

# ⚠️  Make sure the column order and preprocessing match training
#     (e.g., same dummies, same scaling).  If you trained directly on
#     the cleaned CSV without extra transforms, loading the file is
#     enough.  Otherwise apply the same preprocessing pipeline here.

# ------------------------------------------------------------------
# 3.  Predict labels and/or probabilities --------------------------
# ------------------------------------------------------------------
y_pred       = ensemble.predict(X_test)
y_proba      = ensemble.predict_proba(X_test)   # shape = (n_samples, 28)

# ------------------------------------------------------------------
# 4.  Export predictions to disk -----------------------------------
# ------------------------------------------------------------------
out = pd.DataFrame({
        "id"   : X_test.index,     # or any identifier column you have
        "pred" : y_pred
})
out.to_csv("test_predictions.csv", index=False)
print("Wrote test_predictions.csv")

# Optional: if you also need class‑probabilities
proba_df = pd.DataFrame(y_proba, columns=[f"class_{c}"
                                          for c in ensemble.classes_])
proba_df.insert(0, "id", X_test.index)
proba_df.to_csv("test_pred_proba.csv", index=False)
print("Wrote test_pred_proba.csv")



Model saved to vote_ensemble.pkl
Wrote test_predictions.csv
Wrote test_pred_proba.csv


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, log_loss
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier

# ---------------------------
# 1. Define a Cost-Sensitive Wrapper for XGBoost
# ---------------------------
class CostSensitiveXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        # Compute balanced sample weights.
        sample_weight = compute_sample_weight(class_weight='balanced', y=y)
        return super().fit(X, y, sample_weight=sample_weight, **kwargs)

# ---------------------------
# 2. Define Tuning Functions for Each Model
# ---------------------------
def tune_lr(X, y, cw='balanced'):
    """
    Tune logistic regression using GridSearchCV.
    The pipeline uses BorderlineSMOTE followed by logistic regression.
    Returns the best tuned logistic regression pipeline.
    """
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Build pipeline: oversample then fit logistic regression.
    pipeline = ImbPipeline(steps=[
        ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
        ('lr', LogisticRegression(max_iter=2000, class_weight=cw, random_state=42))
    ])
    
    # Grid for logistic regression parameter C.
    param_grid = {
        'lr__C': np.logspace(-2, 2, 10)
    }
    
    grid = GridSearchCV(pipeline, param_grid=param_grid, cv=kf,
                        scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X, y)
    best_pipeline = grid.best_estimator_
    print(f"\n[LR] Best CV accuracy: {grid.best_score_:.4f}")
    print("[LR] Best parameters:", grid.best_params_)
    return best_pipeline

def tune_lgbm(X, y, cw='balanced'):
    """
    Tune LightGBM using GridSearchCV.
    The pipeline uses BorderlineSMOTE then LightGBM.
    Returns the best tuned LightGBM pipeline.
    """
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    pipeline = ImbPipeline(steps=[
        ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
        ('lgbm', LGBMClassifier(n_estimators=100, class_weight=cw, random_state=42))
    ])
    
    param_grid = {
        'lgbm__n_estimators': [50, 100, 200],
        'lgbm__learning_rate': [0.01, 0.1]
    }
    
    grid = GridSearchCV(pipeline, param_grid=param_grid, cv=kf,
                        scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X, y)
    best_pipeline = grid.best_estimator_
    print(f"\n[LGBM] Best CV accuracy: {grid.best_score_:.4f}")
    print("[LGBM] Best parameters:", grid.best_params_)
    return best_pipeline

def tune_xgb(X, y, cw='balanced'):
    """
    Tune XGBoost using GridSearchCV.
    The pipeline uses BorderlineSMOTE then a cost-sensitive XGBoost.
    Returns the best tuned XGBoost pipeline.
    """
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    pipeline = ImbPipeline(steps=[
        ('bsmote', BorderlineSMOTE(random_state=42, k_neighbors=3)),
        ('xgb', CostSensitiveXGBClassifier(n_estimators=100,
                                           objective='multi:softprob',
                                           num_class=len(np.unique(y)),
                                           use_label_encoder=False,
                                           eval_metric='mlogloss',
                                           random_state=42,
                                           class_weight=cw))
    ])
    
    param_grid = {
        'xgb__n_estimators': [50, 100],
        'xgb__max_depth': [3, 5],
        'xgb__learning_rate': [0.01, 0.1]
    }
    
    grid = GridSearchCV(pipeline, param_grid=param_grid, cv=kf,
                        scoring='accuracy', n_jobs=-1, verbose=1)
    grid.fit(X, y)
    best_pipeline = grid.best_estimator_
    print(f"\n[XGB] Best CV accuracy: {grid.best_score_:.4f}")
    print("[XGB] Best parameters:", grid.best_params_)
    return best_pipeline

# ---------------------------
# 3. Load and Split the Data
# ---------------------------
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')
y = y['label']  # assuming label column is 'label'

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

# ---------------------------
# 4. Tune Each Model Separately and Save Them
# ---------------------------
print("\nTuning Logistic Regression...")
best_lr = tune_lr(X_train, y_train, cw='balanced')
joblib.dump(best_lr, 'best_lr.pkl')

print("\nTuning LightGBM...")
best_lgbm = tune_lgbm(X_train, y_train, cw='balanced')
joblib.dump(best_lgbm, 'best_lgbm.pkl')

print("\nTuning XGBoost...")
best_xgb = tune_xgb(X_train, y_train, cw='balanced')
joblib.dump(best_xgb, 'best_xgb.pkl')

# ---------------------------
# 5. Build the Ensemble from the Best Models and Train
# ---------------------------
ensemble = VotingClassifier(
    estimators=[
        ('lr', best_lr),
        ('lgbm', best_lgbm),
        ('xgb', best_xgb)
    ],
    voting='soft'  # average predicted probabilities
)
ensemble.fit(X_train, y_train)

# ---------------------------
# 6. Evaluate the Ensemble on the Validation Set
# ---------------------------
y_pred = ensemble.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print("\nEnsemble Validation Accuracy:", acc)
print("\nEnsemble Classification Report:")
print(classification_report(y_val, y_pred))

# ---------------------------
# 7. Define Function to Compute Weighted Log Loss
# ---------------------------
def weighted_log_loss_from_labels(y_true, y_pred_proba, n_classes):
    """
    Compute weighted log loss where each sample's loss is weighted by
    the inverse of its class frequency (normalized so that weights sum to 1).
    """
    # One-hot encode the true labels
    y_true_ohe = (np.arange(n_classes) == np.array(y_true)[:, None]).astype(int)
    # Compute class frequency counts and corresponding weights
    class_counts = np.sum(y_true_ohe, axis=0)
    class_weights = 1.0 / class_counts
    # Normalize weights to sum to 1
    class_weights /= np.sum(class_weights)
    # Compute per-sample weights from the one-hot representation
    sample_weights = np.sum(y_true_ohe * class_weights, axis=1)
    # Compute log loss for each sample (avoid log(0) with epsilon)
    eps = 1e-15
    log_losses = -np.sum(y_true_ohe * np.log(y_pred_proba + eps), axis=1)
    return np.mean(sample_weights * log_losses)

# ---------------------------
# 8. Compute and Print the Weighted Average Loss on the Validation Set
# ---------------------------
y_val_proba = ensemble.predict_proba(X_val)
weighted_loss = weighted_log_loss_from_labels(y_val, y_val_proba, num_classes)
print("\nWeighted Average Loss (Validation):", weighted_loss)


Number of classes: 28

Tuning Logistic Regression...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html